In [1]:
import os
os.chdir('/home/hrayrh/projects/incontext_spurious')
os.environ['DATA_ROOT_DIR'] = '/home/hrayrh/data'

In [2]:
import sys
sys.path.append('/home/hrayrh/projects/incontext_spurious')

In [3]:
from collections import Counter
from matplotlib import pyplot as plt
%matplotlib inline
import numpy as np
from sklearn.linear_model import LogisticRegression
from src.datamodules.datasets import CivilCommentsForEncodingExtraction, CivilCommentsExtracted

# Inspect CivilCommentsForEncodingExtraction

In [7]:
ds = CivilCommentsForEncodingExtraction(root_dir='/home/hrayrh/data/')
train = ds.get_subset(split='train')

In [9]:
len(train)

269038

In [12]:
for i in range(10):
    print(train[i])

('even up here.......BLACKS!', 2)
("Blame men.  There's always an excuse to blame men for the failures of women.", 3)
("You have no business making any comments on this site, Craig.  For all the  bigot and racists comments that you've personally posted on this site against rural Alaska Natives, the homeless, and the LGBT community, you should be banned from this site.  Alaska and the nation don't need your kind breathing our air.  No.  It's not time to move on.  Every time you rear your bigot, racist head, I'll be  there to call you down and so should everyone else that is exposed to your garbage.", 5)
('"Let\'s get the black folks and the white folks at each others\' throats"', 6)
('I guess the issue is people not willing to put up with the stupid charade of pretending they are willing to live as brother and sister.', 7)
('JackJohnson5:\n"And they say Trump supporters are uneducated!"\n\n-\n\nNo, not necessarily uneducated, just too emotionally immature to admit that they\'ve bought a

In [11]:
val = ds.get_subset(split='val')
len(val)

45180

# Inspect extracted files

In [13]:
saved_data = np.load('/home/hrayrh/data/civilcomments/bert/test/combined.npz')
saved_data.keys()

KeysView(NpzFile '/home/hrayrh/data/civilcomments/bert/test/combined.npz' with keys: encodings, indices_map)

In [14]:
saved_data['encodings'].shape

(133782, 768)

In [15]:
len(saved_data['indices_map'])

447998

In [16]:
saved_data['indices_map'][:100]

array([ 0, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1,  2, -1, -1,  3, -1,  4,
        5,  6,  7,  8, -1, -1,  9, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, 10, -1, 11, -1, -1, 12, 13, 14, 15, -1, -1, 16, -1, -1,
       17, -1, -1, -1, -1, -1, -1, 18, 19, -1, 20, -1, -1, -1, -1, 21, -1,
       -1, -1, -1, -1, 22, -1, -1, 23, 24, 25, 26, 27, 28, 29, -1, 30, 31,
       32, 33, 34, -1, 35, -1, -1, 36, -1, -1, -1, -1, -1, -1, 37])

In [17]:
saved_data['indices_map'][-100:]

array([133749,     -1, 133750,     -1, 133751,     -1,     -1,     -1,
       133752, 133753,     -1,     -1,     -1, 133754,     -1,     -1,
       133755,     -1,     -1,     -1, 133756,     -1,     -1, 133757,
           -1,     -1,     -1,     -1, 133758,     -1,     -1,     -1,
           -1,     -1, 133759, 133760,     -1,     -1,     -1, 133761,
       133762,     -1, 133763,     -1, 133764, 133765, 133766, 133767,
           -1,     -1, 133768,     -1,     -1, 133769,     -1,     -1,
           -1,     -1,     -1, 133770, 133771,     -1,     -1,     -1,
           -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,
           -1,     -1,     -1,     -1,     -1, 133772, 133773,     -1,
           -1, 133774,     -1, 133775, 133776, 133777,     -1,     -1,
       133778,     -1,     -1, 133779,     -1,     -1,     -1, 133780,
           -1,     -1,     -1, 133781])

# Inspecting CivilCommentsExtracted to see how well Y and S are predictable from encodings


**Bert**

512 examples
* Y acc = 87.4
* S acc = 96.5

full dataset (270k examples)
* Y acc@full = 89.9
* S acc@full = 97.7



In [20]:
ds = CivilCommentsExtracted(
    root_dir='/home/hrayrh/data',
    encoding_extractor='bert',
    # encoding_extractor='stroberta',
    # encoding_extractor='roberta',
    # encoding_extractor='roberta_large',
)

In [21]:
train = ds.get_subset('train')
val = ds.get_subset('val')
test = ds.get_subset('test')
print(f'{len(train)=}, {len(val)=}, {len(test)=}')

len(train)=269038, len(val)=45180, len(test)=133782


In [28]:
def compute_group_proportions(examples):
    g = 2 * examples[:, 2] + examples[:, 1]  # 2 * y + c
    return [np.mean(g==i) for i in range(4)]

In [29]:
train_X, examples = train[np.arange(len(train))]
print(f'{train_X.shape=}, {examples.shape=}')
train_s = examples[:, 1]
train_y = examples[:, 2]
print(f'{train_s.shape=}, {train_y.shape=}')
print(compute_group_proportions(examples))

train_X.shape=(269038, 768), examples.shape=(269038, 3)
train_s.shape=(269038,), train_y.shape=(269038,)
[0.8613578750957114, 0.025219485723206387, 0.10185921691359585, 0.011563422267486377]


In [30]:
val_X, examples = val[np.arange(len(val))]
print(f'{val_X.shape=}, {examples.shape=}')
val_s = examples[:, 1]
val_y = examples[:, 2]
print(f'{val_s.shape=}, {val_y.shape=}')
print(compute_group_proportions(examples))

val_X.shape=(45180, 768), examples.shape=(45180, 3)
val_s.shape=(45180,), val_y.shape=(45180,)
[0.8633466135458168, 0.024767596281540506, 0.10008853474988934, 0.01179725542275343]


In [31]:
test_X, examples = test[np.arange(len(test))]
print(f'{test_X.shape=}, {examples.shape=}')
test_s = examples[:, 1]
test_y = examples[:, 2]
print(f'{test_s.shape=}, {test_y.shape=}')
print(compute_group_proportions(examples))

test_X.shape=(133782, 768), examples.shape=(133782, 3)
test_s.shape=(133782,), test_y.shape=(133782,)
[0.8612743119403208, 0.02492861520981896, 0.10230823279663931, 0.011488840053220912]


In [27]:
examples[:20]

array([[ 0,  0,  1],
       [ 1,  0,  0],
       [ 2,  0,  0],
       [ 3,  0,  1],
       [ 4,  0,  0],
       [ 5,  0,  1],
       [ 6,  0,  1],
       [ 7,  0,  0],
       [ 8,  0,  1],
       [ 9,  1,  1],
       [10,  0,  1],
       [11,  0,  1],
       [12,  0,  0],
       [13,  0,  1],
       [14,  0,  0],
       [15,  0,  0],
       [16,  1,  1],
       [17,  0,  1],
       [18,  0,  1],
       [19,  0,  0]])

## Predicting Y

In [34]:
log_reg = LogisticRegression(max_iter=10000)
compute_group_proportions(examples)

# selected_indices = np.random.choice(np.arange(len(train_X)), 512, replace=False)
# log_reg.fit(train_X[selected_indices], train_y[selected_indices])

log_reg.fit(train_X, train_y)

val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_y)
print(f'predicting Y, {val_acc=}')

predicting Y, val_acc=0.899203187250996


## Predicting S

In [35]:
log_reg = LogisticRegression(max_iter=10000)

# selected_indices = np.random.choice(np.arange(len(train_X)), 512, replace=False)
# log_reg.fit(train_X[selected_indices], train_s[selected_indices])

log_reg.fit(train_X, train_s)

val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_s)
print(f'predicting S, {val_acc=}') 

predicting S, val_acc=0.9799247454625941


## Imbalanced contexts

In [40]:
def worst_group_acc(pred, y, s):
    g = 2 * y + s
    accs = []
    for g_idx in range(4):
        mask = (g == g_idx)
        g_acc = np.mean(pred[mask] == y[mask])
        accs.append(g_acc)
    print('group accuracies:', accs)
    print('overall accuracy:', np.mean(pred==y))
    return np.min(accs)

In [74]:
train_X, examples = train[np.arange(len(train))]
train_s = examples[:, 1]
train_y = examples[:, 2]
g = 2 * examples[:, 2] + examples[:, 1]  # 2 * y + c

target_group_proportions = [0.45, 0.05, 0.05, 0.45]
current_group_proportions = compute_group_proportions(examples)
current_group_counts = len(train) * np.array(current_group_proportions)
target_group_counts = len(train) * np.array(target_group_proportions)
c = np.min(current_group_counts / target_group_counts)
actual_group_counts = c * target_group_counts / 7  # this results in about 1000 total examples, same as for 512 context_class_size

In [75]:
print(current_group_counts)
print(target_group_counts)
print(actual_group_counts)

[231738.   6785.  27404.   3111.]
[121067.1  13451.9  13451.9 121067.1]
[444.42857143  49.38095238  49.38095238 444.42857143]


In [76]:
train_mask = np.zeros(len(train), dtype=np.bool_)
g = 2 * examples[:, 2] + examples[:, 1]  # 2 * y + c

for i in range(4):
    g_indices = np.where(g == i)[0]
    selected_g_indices = np.random.choice(
        g_indices,
        int(actual_group_counts[i]),
        replace=False
    )
    train_mask[selected_g_indices] = True

print(np.sum(train_mask))
print(np.sum(actual_group_counts))

986
987.6190476190477


In [82]:
# normal ERM
log_reg = LogisticRegression(max_iter=1000)    
log_reg.fit(train_X[train_mask], train_y[train_mask])

# downsampled ERM
if False:
    log_reg = LogisticRegression(max_iter=1000)    
    X = train_X[train_mask]
    y = train_y[train_mask]
    s = train_s[train_mask]
    g = 2 * y + s
    counts = Counter(g)
    print(counts)
    min_count = np.min([counts[g_idx] for g_idx in counts])
    selected_indices = []
    for g_idx in range(4):
        selected_indices.append(
            np.random.choice(np.where(g==g_idx)[0], size=min_count, replace=False)
        )
    selected_indices=np.concatenate(selected_indices, axis=0)
    log_reg.fit(X[selected_indices], y[selected_indices])

In [83]:
val_pred = log_reg.predict(val_X)
print(f'predicting Y, matching Y, wg acc: {worst_group_acc(val_pred, val_y, val_s)}')

group accuracies: [0.8714556734861303, 0.28328865058087577, 0.40026536930561696, 0.8836772983114447]
overall accuracy: 0.8098716246126605
predicting Y, matching Y, wg acc: 0.28328865058087577


In [84]:
val_pred = log_reg.predict(val_X)
print(f'predicting Y, matching S, wg acc: {worst_group_acc(val_pred, val_s, val_y)}')

group accuracies: [0.8714556734861303, 0.599734630694383, 0.7167113494191242, 0.8836772983114447]
overall accuracy: 0.8405710491367862
predicting Y, matching S, wg acc: 0.599734630694383


# Inspect CivilCommentsEmbContextsDatasetV2

In [4]:
from src.datamodules.datasets import CivilCommentsEmbContextsDatasetV2

In [14]:
ds = CivilCommentsEmbContextsDatasetV2(
    root_dir='/home/hrayrh/data/',
    encoding_extractor='bert',
    data_length=1000,
    context_split='train',
    query_split='val',
    context_class_size=256,
    context_group_proportions=[0.45, 0.05, 0.05, 0.45],
    query_group_proportions=[0.25, 0.25, 0.25, 0.25],
    spurious_setting='wb_dro',
    sp_token_generation_mode='opposite',
    use_context_as_intermediate_queries=False,
    permute_input_dim=True,
    simpler_construction=True,
)

In [15]:
input_seq, context, queries, query_indices = ds[0]

In [16]:
input_seq.shape

(1024, 768)

In [17]:
context.shape

(512, 3)

In [18]:
queries.shape

(512, 3)

In [19]:
query_indices[:10]

array([ 1,  3,  5,  7,  9, 11, 13, 15, 17, 19])

In [20]:
input_seq[0][:10]

array([-1.        , -1.        , -1.        , -0.28471977,  0.01991533,
       -0.2839902 , -0.19192772,  0.28720826,  0.16731384,  0.13958192],
      dtype=float32)

In [21]:
input_seq[1][:10]

array([ 0.        ,  0.        ,  1.        , -0.11815555, -0.3563908 ,
       -0.39131027,  0.00670241,  0.54735464, -0.1141782 ,  0.10982314],
      dtype=float32)

In [39]:
for i in range(0, 50, 2):
    print(input_seq[i, 0], input_seq[i, 1], input_seq[i, 2], context[i//2, 2], context[i//2, 1])

-1.0 -1.0 -1.0 0 0
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
1.0 1.0 -1.0 1 1
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
1.0 1.0 -1.0 1 1
1.0 1.0 -1.0 1 1
1.0 -1.0 -1.0 1 0
1.0 1.0 -1.0 1 1
-1.0 -1.0 -1.0 0 0
-1.0 -1.0 -1.0 0 0
-1.0 -1.0 -1.0 0 0


In [40]:
for i in range(1, 50, 2):
    print(input_seq[i, 0], input_seq[i, 1], input_seq[i, 2])

0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
0.0 0.0 1.0
